# 6.6 Roos and Louis in French: a high-dimensional picture from trigrams

> **Question:** when we write to each other in French, do Louis and I write *differently*,
> and can you see that in the characters we type?

**My expectation** *(to fill in myself, before reading the results)*: …

Louis is a native speaker; French is my second language. That predicts a difference, but it
does not say what kind: different words, different spelling, different punctuation, or a
different phone keyboard. This notebook is 6.2's method on my own chat, French only:

1. **Rows:** every French message from both of us, all dates, cut into chunks of equal length
   per author.
2. **Columns:** character trigrams, counted (6.2.3). Thousands of columns: the high-dimensional
   part.
3. **Picture:** Manhattan distances between chunks, squashed to two dimensions with PCA (6.1,
   6.2.4).
4. **Number:** the silhouette by author against shuffled labels, then *which* trigrams do the
   separating, then 6.2.5's check: does it survive with only letters left?

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.datatransforms import Pipeline
from goad_toolkit.visualizer import PlotSettings, ProjectionPlot, ScreePlot
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import manhattan_distances

from roos_pipeline import settings
from roos_pipeline.settings import Paths
from roos_pipeline.vectors import StripQuotedReply

AUTHOR_COLORS = {"Roos": settings.ROOS_COLOR, "Louis": settings.LOUIS_COLOR}
PERIOD_COLORS = {"before move": settings.LONG_DISTANCE_ORANGE, "after move": settings.TOGETHER_BLUE}

## 6.6.1 What a trigram is

A trigram is a window of three characters that slides over the text one step at a time,
spaces and punctuation included. "oui ça va" gives `oui`, `ui `, `i ç`, ` ça`, `ça `, `a v`,
` va`. Counting them turns a text into a row of numbers: one column per trigram that occurs
anywhere, and in each cell how often this text contains it.

Two short French sentences, written the way each of us might write them:

In [ ]:
example = ["oui ça va, je t'aime", "Oui ca va ? Je t’aime"]
toy = CountVectorizer(analyzer="char", ngram_range=(3, 3))
toy_counts = pd.DataFrame(toy.fit_transform(example).toarray(), index=pd.Index(example),
                          columns=toy.get_feature_names_out())
print(f"{toy_counts.shape[1]} distinct trigrams")
toy_counts.loc[:, toy_counts.nunique() > 1]  # only the columns where the two differ

The same words, and still a dozen columns differ: `ça` against `ca`, the straight `'` against
the curly `’` a phone keyboard types, and the space before `?` that French typography asks for.
A trigram counter sees *how* something is typed, not only *what* is said. That is its strength
for telling writers apart, and its weakness: a phone setting looks the same to it as a habit.
(`CountVectorizer` lowercases by default, so `Oui` and `oui` are one column.)

## 6.6.2 The data: every French message, from both of us

From `2_features.parquet`, which carries `language` (`py3langid`, see 05.6). All dates, not
only before Benji. `StripQuotedReply` first (a quoted reply would put one person's words in the
other's mouth), then links out: a pasted link is not typed (6.2.4).

In [ ]:
features = pd.read_parquet(Paths().features)
french = features[features["language"] == "fr"].sort_values(["author", "ts_local"])
french = Pipeline().add(StripQuotedReply).apply(french.copy())
french["text"] = (french["message"].str.lower()
                  .str.replace(settings.LINK_PATTERN, " ", regex=True)
                  .str.replace(r"\s+", " ", regex=True).str.strip())
french = french[french["text"].str.len() >= settings.MIN_CHARS]

french.groupby("author").agg(
    messages=("text", "size"),
    characters=("text", lambda s: s.str.len().sum()),
    median_length=("text", lambda s: s.str.len().median()),
    first=("ts_local", "min"),
    last=("ts_local", "max"),
)

**Two things to keep in mind before any picture.** I wrote about six times as much French as
Louis. And not at the same time: most of my French is from the long-distance months, most of
his from after the move (05.6). So "Roos against Louis" is partly "before the move against
after". 6.6.4 checks whether the difference survives within one period.

## 6.6.3 Chunks of equal length

One message is too short to say anything (median a few dozen characters, almost all zeros in
the count). So, as in 6.2: consecutive messages of one author are joined into chunks of
`settings.TRIGRAM_CHUNK_CHARS` characters. Equal length matters, because Manhattan distance
counts differences in *counts*, and a longer chunk has bigger counts.

Then the chunks are **balanced**: as many of mine as Louis has, drawn at random. With six times
more of my chunks, the picture and the silhouette would mostly describe me.

In [ ]:
def french_chunks(frame: pd.DataFrame, size: int, seed: int = settings.SEED) -> pd.DataFrame:
    # Consecutive messages of one author, joined into chunks of about `size` characters; the
    # last, short remainder is dropped; then an equal number of chunks per author.
    running = frame["text"].str.len().add(1).groupby(frame["author"]).cumsum()  # +1: the joining space
    chunks = (frame.assign(chunk=running // size)
              .groupby(["author", "chunk"])
              .agg(text=("text", " ".join), n_messages=("text", "size"),
                   start=("ts_local", "min"), period=("move_period", lambda s: s.mode().iloc[0]))
              .reset_index())
    chunks = chunks[chunks["text"].str.len() > size / 2]
    fewest = chunks["author"].value_counts().min()
    return chunks.groupby("author").sample(n=fewest, random_state=seed).reset_index(drop=True)


chunks = french_chunks(french, settings.TRIGRAM_CHUNK_CHARS)
print(f"{len(chunks)} chunks of ~{settings.TRIGRAM_CHUNK_CHARS:,} characters")
chunks.groupby(["author", "period"]).agg(chunks=("text", "size"), median_messages=("n_messages", "median"))

One chunk holds about twice as many of Louis's messages as of mine: his messages are shorter.
That will show up in the trigrams too, since every message boundary adds a space.

## 6.6.4 Count, measure, look

Every trigram in the French of both of us becomes a column: that is the high-dimensional space.
Each chunk is a point in it. Manhattan distance between every pair of chunks, then PCA on the
rows of that distance matrix (6.2.4), two components.

In [ ]:
def trigram_space(chunks: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, CountVectorizer]:
    # Trigram counts per chunk, and the Manhattan distance between every pair of chunks.
    vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 3))
    counts = vectorizer.fit_transform(chunks["text"]).toarray()
    return counts, manhattan_distances(counts, counts), vectorizer


counts, distance, vectorizer = trigram_space(chunks)
print(f"{counts.shape[0]} chunks x {counts.shape[1]:,} trigrams; a chunk uses {np.mean(counts > 0):.0%} of them")

pca = PCA(n_components=10).fit(distance)
coords = pca.transform(distance)[:, :2]
fig, ax = ScreePlot(PlotSettings(figsize=(7, 3.5), title="How much of the distances two components hold",
                                 xlabel="component", ylabel="share of variance")).plot(
    explained_variance_ratio=pca.explained_variance_ratio_)

First the number, before the picture: a projection always fills its frame (6.1), so the
number that does not depend on the picture:
the silhouette by author on the full distance matrix, against the same score with shuffled
author labels. And the same within each period, so time cannot do the work.

In [ ]:
labels = chunks["author"].to_numpy()
periods = chunks["period"].to_numpy()


def separation(distance: np.ndarray, labels: np.ndarray, seed: int = settings.SEED) -> tuple[float, float]:
    # Silhouette of the labels on a precomputed distance matrix, and the mean for shuffled labels.
    rng = np.random.default_rng(seed)
    real = silhouette_score(distance, labels, metric="precomputed")
    shuffled = np.mean([silhouette_score(distance, rng.permutation(labels), metric="precomputed")
                        for _ in range(settings.N_LABEL_SHUFFLES)])
    return float(real), float(shuffled)


def separation_table(distance: np.ndarray, labels: np.ndarray, periods: np.ndarray) -> pd.DataFrame:
    # The author silhouette over all chunks, and within each period on its own.
    rows = {"all chunks": np.full(len(labels), True)} | {f"only {p}": periods == p for p in settings.PERIOD_ORDER}
    table = {}
    for name, keep in rows.items():
        real, shuffled = separation(distance[np.ix_(keep, keep)], labels[keep])
        table[name] = {"chunks Roos": int((labels[keep] == "Roos").sum()),
                       "chunks Louis": int((labels[keep] == "Louis").sum()),
                       "silhouette by author": round(real, 3), "shuffled labels": round(shuffled, 3)}
    return pd.DataFrame(table).T


separation_table(distance, labels, periods)

Write the share of the first two components in the caption (06.1, "What to write down"); the
first component takes most of it, which 6.6.6 comes back to.

Two panels of the *same* points: on the left coloured by who wrote the chunk, on the right by
period. If the left panel splits and the right panel splits the same way, the picture cannot
tell author from time. The shuffle test from the table above is written into the left panel,
so the picture never travels without its number.

In [ ]:
def write_shuffle_test(ax, distance: np.ndarray, labels: np.ndarray) -> None:
    # The author silhouette and its shuffled baseline, as text in the top-left corner of `ax`.
    real, shuffled = separation(distance, labels)
    ax.text(0.02, 0.98, f"silhouette by author: {real:.3f}\n"
                        f"shuffled labels ({settings.N_LABEL_SHUFFLES}x): {shuffled:.3f}",
            transform=ax.transAxes, ha="left", va="top", fontsize=9,
            bbox={"facecolor": "white", "edgecolor": settings.LIGHT_GREY, "boxstyle": "round"})


explained = pca.explained_variance_ratio_[:2].sum()

panels = PlotSettings(
    figsize=(12, 5.5),
    title=f"French chunks of {settings.TRIGRAM_CHUNK_CHARS:,} characters by trigram counts "
          f"(PCA, {explained:.0%} of the variance)",
    subplot_titles=["who wrote it", "when (before or after the move)"],
    xlabel="",
    ylabel="",
    max_cols=2,
)
host = ProjectionPlot(panels)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(ProjectionPlot(panels), axes[0], coordinates=coords, labels=labels,
                  palette=AUTHOR_COLORS, s=18)
write_shuffle_test(axes[0], distance, labels)
_ = host.plot_on_axes(ProjectionPlot(panels), axes[1], coordinates=coords, labels=periods,
                      palette=PERIOD_COLORS, s=18)

## 6.6.5 Which trigrams do the separating

Counts can be read back (6.2.4). Per trigram: its share of my chunks minus its share of Louis's.

In [ ]:
def trigram_difference(counts: np.ndarray, labels: np.ndarray, vectorizer: CountVectorizer) -> pd.Series:
    # Per trigram: mean share in Roos's chunks minus mean share in Louis's chunks.
    share = counts / counts.sum(axis=1, keepdims=True)
    return pd.Series(share[labels == "Roos"].mean(axis=0) - share[labels == "Louis"].mean(axis=0),
                     index=vectorizer.get_feature_names_out())


difference = trigram_difference(counts, labels, vectorizer)
print("more mine: ", [repr(t) for t in difference.nlargest(15).index])
print("more Louis:", [repr(t) for t in difference.nsmallest(15).index])

The trigrams are fragments, so turn them back into habits and count those directly, per
message. Each row below is a guess from the lists above, checked on the messages themselves:

In [ ]:
HABITS = {
    "writes 'ça' with ç": (r"\bça\b", r"\b[cç]a\b"),
    "'ca' without ç": (r"\bca\b", r"\b[cç]a\b"),
    "curly apostrophe ’": (r"’", r"['’]"),
    "space before ? or !": (r" [?!]", r"[?!]"),
    "says 'oui'": (r"\boui\b", r""),
    "'un peu'": (r"\bun peu\b", r""),
}


def habit_table(frame: pd.DataFrame) -> pd.DataFrame:
    # Per author: of the messages that could show the habit (second pattern), the share that do.
    rows = {}
    for habit, (shows, could) in HABITS.items():
        rows[habit] = {}
        for author, messages in frame.groupby("author")["message"]:
            base = messages[messages.str.contains(could, case=False, regex=True)] if could else messages
            rows[habit][author] = f"{base.str.contains(shows, case=False, regex=True).mean():.0%} of {len(base):,}"
    return pd.DataFrame(rows).T


habit_table(french)

Read every row with its base ("of N"): a share of 30 messages is not a share of 3,000.

## 6.6.6 Check: letters only

Punctuation, emoji, digits and the apostrophe a keyboard types are habits too, but of
*typing*, not of language. 6.2.5's check: keep only letters (accents kept, so `ça` survives) and run
exactly the same steps again.

In [ ]:
letters = french.assign(text=french["text"].str.replace(settings.NOT_A_LETTER_PATTERN, " ", regex=True)
                        .str.replace(r"\s+", " ", regex=True).str.strip())
letters = letters[letters["text"].str.len() >= settings.MIN_CHARS]
letter_chunks = french_chunks(letters, settings.TRIGRAM_CHUNK_CHARS)
letter_counts, letter_distance, letter_vectorizer = trigram_space(letter_chunks)
letter_labels = letter_chunks["author"].to_numpy()

letter_pca = PCA(n_components=2).fit(letter_distance)
passes = PlotSettings(
    figsize=(12, 5.5),
    title="The same French messages, counted twice",
    subplot_titles=["everything (links removed)", "letters only"],
    xlabel="",
    ylabel="",
    max_cols=2,
)
host = ProjectionPlot(passes)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(ProjectionPlot(passes), axes[0], coordinates=coords, labels=labels,
                  palette=AUTHOR_COLORS, s=18, legend=False)
write_shuffle_test(axes[0], distance, labels)
host.plot_on_axes(ProjectionPlot(passes), axes[1], coordinates=letter_pca.transform(letter_distance),
                  labels=letter_labels, palette=AUTHOR_COLORS, s=18)
write_shuffle_test(axes[1], letter_distance, letter_labels)

separation_table(letter_distance, letter_labels, letter_chunks["period"].to_numpy())

In [ ]:
letter_difference = trigram_difference(letter_counts, letter_labels, letter_vectorizer)
print("more mine: ", [repr(t) for t in letter_difference.nlargest(15).index])
print("more Louis:", [repr(t) for t in letter_difference.nsmallest(15).index])

### What is the first component?

It holds most of the variance, so it is worth knowing what it measures before reading the
picture left to right. Two candidates that have nothing to do with style: how many messages
are in a chunk (short messages mean more spaces and more "oui"), and when it was written.

In [ ]:
first = coords[:, 0]
for name, values in [("messages in the chunk", chunks["n_messages"]),
                     ("date of the chunk", chunks["start"].astype("int64"))]:
    rho = spearmanr(first, values).statistic
    print(f"first component vs. {name:22s} spearman {rho:+.2f}")
for author in AUTHOR_COLORS:
    mine = labels == author
    print(f"  within {author:5s}: vs. messages {spearmanr(first[mine], chunks['n_messages'][mine]).statistic:+.2f}")

And the second component, which in the picture looks like the one that runs from me to Louis:
cut it at its median, and count who lands on which side.

In [ ]:
upper_half = np.where(coords[:, 1] > np.median(coords[:, 1]), "upper half", "lower half")
print(f"second component: {pca.explained_variance_ratio_[1]:.0%} of the variance")
pd.crosstab(labels, upper_half)

## What I found

*(To write myself. Questions to answer, each in my own words:)*

1. **How big is the difference?** The silhouette by author against the shuffled one, all chunks
   and within each period. Is it "clearly apart" or "slightly apart"? Compare with 6.2's Q
   chunks (0.03–0.08).
2. **Is it time?** Does the author difference survive within *before the move* and within
   *after the move*?
3. **Is it my phone?** What changed between "everything" and "letters only", in the picture and
   in the number?
4. **What kind of difference is it?** From the trigram lists and the habits table: spelling,
   punctuation, word choice, message length? Which of these would I call *communication*, and
   which is just typing?
5. **What does the first component measure**, and what does that mean for reading the picture?
   Why can the picture look clearly split while the silhouette is small? (Hint: the silhouette
   uses all 11,000 dimensions; the picture shows two, and the split sits in the second.)
6. **What I would write in a caption:** chunk size, chunks per author, the variance in the two
   components, and the claim (do the chunks group by author, yes or no; not where the groups sit).

## 6.6.7 A style map: have we grown towards each other's French?

A second picture, with different choices from 6.6.1–6.6.6, all of them in
[`scripts/stijlkaart.py`](../../scripts/stijlkaart.py) (temporary home; it moves to
`roos_pipeline/` once I approve the figure):

- **French** is decided by stopwords here (≥ 60% of the recognised ones French), not by
  `py3langid`, so the counts differ from 6.6.2.
- **Topic is masked**: every word that is not one of ~150 French function words becomes `x`.
  What is left is style: function words, punctuation, spacing.
- **Time is in it**: per person per quarter 12 random documents of 40 fragments (~600 words),
  tf-idf on trigrams within words (`char_wb`), PCA(2), and the mean per half-year as a path.

In [ ]:
from goad_toolkit.datatransforms import Pipeline

from roos_pipeline.places import UnmergeMessages
from roos_pipeline.prepare import CanonicalAuthor, load_author_aliases
from scripts import stijlkaart as sk

raw = pd.read_parquet(Paths().messages)
messages = (Pipeline()
            .add(UnmergeMessages)  # rows the export glued together, split back into messages
            .add(CanonicalAuthor, aliases=load_author_aliases(Paths().config))
            .add(sk.StripWhatsAppNoise)  # calls, "weggelaten", edits, deletions, URLs; ' for ’
            .add(sk.FrenchByStopwords)
            .apply(raw))
# UnmergeMessages also brings back quoted messages a second time: keep each message once
messages = messages.drop_duplicates(["ts_local", "author", "message"])

units = sk.french_units(messages)
frags = sk.fragments(units)
frags["masked"] = frags["text"].map(sk.mask_content_words)
frags["quarter"] = sk.style_quarter(frags["ts_local"])

print("share of messages that count as French:", messages.groupby("author")["is_french"].mean().round(2).to_dict())
print("units of 8+ words:", units.groupby("author").size().to_dict())
print("fragments of 15 words:", frags.groupby("author").size().to_dict())
frags.groupby(["quarter", "author"]).size().unstack()

In the first quarters Louis has only ~65 fragments: 12 documents of 40 drawn from 65 overlap a
lot, so his early dots are less independent than they look. What masking does to a fragment:

In [ ]:
frags.sample(4, random_state=settings.SEED)[["text", "masked"]]

In [ ]:
docs = sk.sample_documents(frags)
vectorizer, pca, doc_coords = sk.fit_style_space(docs)
print(f"{len(docs)} documents x {len(vectorizer.vocabulary_)} trigrams; "
      f"the two axes hold {pca.explained_variance_ratio_.sum():.0%} of the variance "
      f"(x {pca.explained_variance_ratio_[0]:.0%}, y {pca.explained_variance_ratio_[1]:.0%})")
for axis, ends in sk.axis_loadings(vectorizer, pca, n=15).items():
    print(f"\n{axis} low : {ends['low']}\n{axis} high: {ends['high']}")

**Check the axis direction.** The labels say: y from school French (*beaucoup*, *est-ce*, *n'*) up
to spoken French (*ca*, *faut*, *oui*, a space before *?*); x from correct French (*doit*, *veux*,
*chez*) to loose punctuation and English leftovers. PCA's sign is arbitrary, so an axis whose
markers point the wrong way is flipped (× −1). The markers are trigrams from the lists above.

In [ ]:
MARKERS = {
    "x": {"high": (" ;)", " : ", " ! ", " or"), "low": ("doi", "veu", "hez", "peu")},
    "y": {"high": (" ca", "ca ", "fau", "oui", " ? "), "low": ("bea", "uco", "-ce", " n'")},
}
signs = tuple(int(np.sign(sk.marker_score(vectorizer, component, **MARKERS[axis])))
              for axis, component in zip(("x", "y"), pca.components_))
print("signs (x, y):", signs, "(-1 = flipped)")
doc_coords = sk.orient_axes(doc_coords, signs)
means = sk.half_year_means(docs, doc_coords)
means.round({"x": 3, "y": 3})

### Tests

- **Trend**: every fragment projected onto the two axes; per person per month the mean position
  (months with ≥ 10 fragments); Spearman between month and position, Roos on y, Louis on x;
  p from 10,000 shuffles of the months.
- **Similarity**: per quarter the cosine between Roos's and Louis's mean tf-idf profile, with a
  Spearman trend over the quarters and the same permutation test.

In [ ]:
frag_coords = sk.orient_axes(pca.transform(vectorizer.transform(frags["masked"]).toarray()), signs)
roos_trend = sk.monthly_trend(frags, frag_coords, "Roos", axis=1)
louis_trend = sk.monthly_trend(frags, frag_coords, "Louis", axis=0)
similarity = sk.quarterly_similarity(docs, vectorizer)
sim_rho, sim_p = sk.permutation_spearman(np.arange(len(similarity)), similarity.to_numpy())

print(f"Roos on y (spoken up):     rho = {roos_trend['rho']:+.2f}, {sk.p_text(roos_trend['p'])}, "
      f"{roos_trend['n_months']} months")
print(f"Louis on x (correct left): rho = {louis_trend['rho']:+.2f}, {sk.p_text(louis_trend['p'])}, "
      f"{louis_trend['n_months']} months")
print(f"similarity over quarters:  rho = {sim_rho:+.2f}, {sk.p_text(sim_p)}, {len(similarity)} quarters")
similarity.round(2)

### The figure

Every sentence on it is in `texts` below, so the wording stays mine to decide. **The title and
subtitle are the draft from the example** and say more than the tests do: the tests show that our
styles moved and became more alike, not *why* ("learned from Louis" is a cause, and this data
cannot show a cause). The examples under the axis words are taken from the loadings above.

In [ ]:
first_q, last_q = similarity.index[0], similarity.index[-1]
used = messages[messages["is_french"]]
# "aug 2023 – sep 2026": lowercase English month abbreviations read as Dutch ones
period = f"{used['ts_local'].min():%b %Y} – {used['ts_local'].max():%b %Y}".lower()
texts = {
    "title": "Roos en Louis groeien naar elkaars Frans toe",
    "subtitle": ("Roos ruilt schoolboek-Frans in voor spreektaal geleerd van Louis, Louis laat zijn Engels los.\n"
                 f"Hun stijlprofielen lijken steeds meer op elkaar: overeenkomst {similarity[first_q]:.2f} in "
                 f"{first_q[:4]}, {similarity[last_q]:.2f} in {last_q[:4]} (1 = identiek)."),
    "x_low": "← correct Frans",
    "x_low_examples": '"doit", "veux", "peut", "chez"',
    "x_high": "Engelse resten en losse leestekens →",
    "x_high_examples": '";)", " : ", " ! ", “ ” « », "or"',
    "y_low": "↓ schoolboek-Frans",
    "y_low_examples": '"beaucoup", "est-ce que", "n\'", ":)"',
    "y_high": "spreektaal ↑",
    "y_high_examples": '"ca", "faut", "oui", "quoi", " ? "',
    "footnote": (
        f"Franse WhatsApp-berichten van Roos en Louis, {period}. Trigrammen = reeksen van 3 tekens; inhoudswoorden "
        "vervangen door \"x\", zodat onderwerp niet meetelt.\n"
        f"Per persoon per kwartaal {sk.DOCS_PER_QUARTER} steekproeven van ±600 woorden → tf-idf "
        f"({len(vectorizer.vocabulary_)} trigrammen) → PCA. De twee assen tonen {pca.explained_variance_ratio_.sum():.0%} van de variatie.\n"
        f"Toets: verschuift de stijl met de tijd? Roos omhoog naar spreektaal ρ = {roos_trend['rho']:.2f}, "
        f"{sk.p_text(roos_trend['p'])} · Louis naar links ρ = {louis_trend['rho']:.2f}, "
        f"{sk.p_text(louis_trend['p'])} · overeenkomst ρ = {sim_rho:.2f}, {sk.p_text(sim_p)}.\n"
        "Spearman-correlatie tussen maand en de gemiddelde positie van alle fragmenten van 15 woorden "
        f"in die maand (Roos {roos_trend['n_months']}, Louis {louis_trend['n_months']} maanden met "
        f"≥{sk.MIN_MONTH_FRAGMENTS} fragmenten);\np-waarde via {sk.N_PERMUTATIONS:,} keer de maanden "
        "door elkaar husselen. Overeenkomst = cosinus tussen de trigramprofielen per kwartaal "
        f"({len(similarity)} kwartalen), zelfde permutatietoets."
    ).replace(f"{sk.N_PERMUTATIONS:,}", f"{sk.N_PERMUTATIONS:,}".replace(",", ".")),  # Dutch 10.000
}

fig = sk.draw_stijlkaart(docs, doc_coords, means, texts)
out = Paths().figures / "frans_van_roos_en_louis.png"
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=150, facecolor=sk.BACKGROUND)
print(f"saved {out}")

### Check: the main numbers

In [ ]:
print("fragments per person:", frags.groupby("author").size().to_dict())
print(f"explained variance: x {pca.explained_variance_ratio_[0]:.1%}, y {pca.explained_variance_ratio_[1]:.1%}, "
      f"together {pca.explained_variance_ratio_.sum():.1%}")
for name, rho, p in [("Roos, y", roos_trend["rho"], roos_trend["p"]),
                     ("Louis, x", louis_trend["rho"], louis_trend["p"]),
                     ("similarity", sim_rho, sim_p)]:
    print(f"{name:11s} rho = {rho:+.2f}, p = {p:.4f}")
print(f"similarity first quarter ({first_q}): {similarity[first_q]:.2f}, last ({last_q}): {similarity[last_q]:.2f}")